### Setup

In [ ]:
import csv
import math
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class ShapeSpec:
    b: int
    c: int
    h: int
    w: int


SIZES         = (4, 8, 16, 32, 64, 128)
SHAPES        = [ShapeSpec(b=b, c=c, h=hw, w=hw) for b in SIZES for c in SIZES for hw in SIZES]
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE         = torch.float32
KERNEL_RADIUS = 2
WARMUP        = 1
RUNS          = 1
MODE          = "mean"
OUTPUT_LOG    = Path(f"redundancy_speed_{MODE}.csv") 

torch.set_printoptions(sci_mode=False)
torch.manual_seed(41)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(41)

print(f"[device]: {DEVICE}")

#### Max's Function

Max's function extracts patches at each dw - dh, and uses a tensor operation to find the batch-wise correlation quickly. It still has to loop over dh, dw, and the channels

In [ ]:
def max_fn(feature_map: torch.Tensor, kernel_radius: int, mode: str = "max") -> torch.Tensor:
    if mode not in {"mean", "max"}:
        raise ValueError(f"mode must be 'mean' or 'max', got {mode!r}")

    eps        = 1e-6
    B, C, H, W = feature_map.shape
    R          = kernel_radius

    # 1) Compute Z-score each coordinate across batch
    mu  = feature_map.mean(dim=0, keepdim=True)
    sig = feature_map.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z   = (feature_map - mu) / sig # [B, C, H, W]

    # 2) Get all the possible shifts within `kernel_radius` distance
    shifts: list[tuple[int, int]] = []
    for dh in range(-R, R + 1):
        for dw in range(-R, R + 1):
            if dh == 0 and dw == 0:
                continue
            shifts.append((dh, dw))

    # 3) Fill out the channel correlation matrix
    corr = torch.zeros((C, C), device=Z.device, dtype=Z.dtype)

    corr_count = None
    if mode == "mean":
        corr_count = torch.zeros((C, C), device=Z.device, dtype=torch.int32)
    elif mode == "max":
        pass

    for dh, dw in shifts:
        mh = H - abs(dh)
        mw = W - abs(dw)
        if mh <= 0 or mw <= 0:
            continue

        h1_start = max(0, -dh)
        h2_start = max(0, dh)
        w1_start = max(0, -dw)
        w2_start = max(0, dw)

        for c1 in range(C):
            for c2 in range(c1, C):
                m1 = Z[:, c1, h1_start : h1_start + mh, w1_start : w1_start + mw]  # [B, mh, mw]
                m2 = Z[:, c2, h2_start : h2_start + mh, w2_start : w2_start + mw]  # [B, mh, mw]

                m1_flat = m1.flatten(1)  # [B, mh * mw]
                m2_flat = m2.flatten(1)  # [B, mh * mw]

                r_flat = torch.einsum("bn,bn->n", m1_flat, m2_flat) / B  # [mh * mw]
                r2_all = r_flat * r_flat

                if mode == "max":
                    r2 = r2_all.amax()
                elif mode == "mean":
                    r2 = r2_all.mean()

                if mode == "mean":
                    corr[c1, c2]       += r2
                    corr[c2, c1]       += r2
                    corr_count[c1, c2] += 1
                    corr_count[c2, c1] += 1
                elif mode == "max":
                    updated = torch.maximum(corr[c1, c2], r2)
                    corr[c1, c2] = updated
                    corr[c2, c1] = updated

    # 4) Remove same-channel pairs
    mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=DEVICE), diagonal=1)

    if mode == "mean":
        corr = corr / corr_count.clamp_min(1).to(dtype=corr.dtype)
        vals = corr.masked_select(mask)
        if vals.numel() == 0:
            return corr.new_zeros(())
        return vals.mean()
    elif mode == "max":
        corr = corr * mask
        max_corr = torch.amax(corr, dim=(0, 1))
        return max_corr

#### Luca's Function

Luca's function extracts patches in each immediate neighborhood of a position, and uses all tensorized operations. This likely comes at the cost of high GPU usage.

In [ ]:
def luca_fn(feature_map: torch.Tensor, kernel_radius: int, mode: str = "max") -> torch.Tensor:
    if mode not in {"mean", "max"}:
        raise ValueError(f"mode must be 'mean' or 'max', got {mode!r}")

    eps        = 1e-6
    B, C, H, W = feature_map.shape
    FM         = feature_map
    R          = kernel_radius
    
    # 1) Z-score each coordinate across batch
    mu  = FM.mean(dim=0, keepdim=True)
    sig = FM.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z   = (FM - mu) / sig # [B, C, H, W]

    # 2) Extract the neighborhood within `kernel_radius` distance of each position
    patch_HW  = (2 * R) + 1
    patch_len = patch_HW * patch_HW
    patches   = F.unfold(Z, kernel_size=patch_HW, padding=R) # [B, C * patch_len, H * W]
    patches   = patches.view(B, C, patch_len, H * W)         # [B, C, patch_len,  H * W]                     

    # 3) The center element in each neighborhood corresponds to (dh=0, dw=0)
    center_k = (patch_len) // 2
    center   = patches[:, :, center_k,        :] # [B, C,                  H * W]
    patches  = patches[:, :, :, :]               # [B, C, patch_len // 2,  H * W]   

    # 4) Build a channel correlation matrix for every c1 (i) and c2 (j)
    corr = torch.einsum('bip,bjkp->ijkp', center, patches) / B # [C1, C2, patch_len // 2, H * W]
    corr = corr * corr                                         # [C1, C2, patch_len // 2, H * W]

    same_ch    = torch.eye(C, dtype=torch.bool, device=corr.device)[:, :, None, None]
    center_pos = (torch.arange(patch_len, device=corr.device) == center_k)[None, None, :, None]
    corr       = corr.masked_fill(same_ch & center_pos, 0)

    if mode == "max":
        corr = corr.amax(dim=(2, 3))
    elif mode == "mean":
        corr = corr.mean(dim=(2, 3))
    
    # 5) Remove same-channel pairs
    mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=DEVICE), diagonal=1)
    corr = corr * mask

    return corr

### Experiments

In [ ]:
def time_fn(
    fn,
    fm: torch.Tensor,
    kernel_radius: int,
    mode: str,
    warmup: int = 2,
    runs: int = 10,
) -> tuple[float, float]:
    # Warm up CUDA
    for _ in range(warmup):
        out = fn(fm, kernel_radius, mode)
    torch.cuda.synchronize()

    times: list[float] = []
    for _ in range(runs):
        t0 = time.perf_counter()
        out = fn(fm, kernel_radius, mode)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        times.append(t1 - t0)

    value = float(out.detach().float().cpu())
    return float(sum(times) / max(1, len(times))), value

In [ ]:
def make_fm(shape: ShapeSpec) -> torch.Tensor:
    return torch.randn(
        (shape.b, shape.c, shape.h, shape.w),
        device=DEVICE,
        dtype=DTYPE,
    )

In [ ]:
def benchmark(
    shapes: list[ShapeSpec],
    kernel_radius: int = 2,
    mode: str = "max",
    warmup: int = 2,
    runs: int = 10,
    output_log: Path | None = Path("redundancy_speed_outputs.csv"),
) -> list[dict]:
    fns = {
        "max_fn": max_fn, 
        "luca_fn": luca_fn
    }

    fieldnames = [
        "kernel_radius",
        "b",
        "c",
        "h",
        "w",
        "max_fn_sec",
        "max_fn_value",
        "luca_fn_sec",
        "luca_fn_value",
    ]

    if output_log is not None:
        output_log.parent.mkdir(parents=True, exist_ok=True)
        with output_log.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    results: list[dict] = []
    for shape in tqdm(shapes):
        fm = make_fm(shape)
        row: dict = {
            "b": shape.b,
            "c": shape.c,
            "h": shape.h,
            "w": shape.w,
        }
        for name, fn in fns.items():
            sec, val = time_fn(fn, fm, kernel_radius=kernel_radius, mode=mode, warmup=warmup, runs=runs)
            row[f"{name}_sec"] = sec
            row[f"{name}_value"] = val


        if output_log is not None:
            with output_log.open("a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow({"kernel_radius": kernel_radius, **row})

        results.append(row)

    return results

In [ ]:
results = benchmark(
    SHAPES,
    kernel_radius=KERNEL_RADIUS,
    mode=MODE,
    warmup=WARMUP,
    runs=RUNS,
    output_log=OUTPUT_LOG,
)

fastest = sorted(results, key=lambda r: min(r["max_fn_sec"], r["luca_fn_sec"]))[:10]
slowest = sorted(results, key=lambda r: max(r["max_fn_sec"], r["luca_fn_sec"]), reverse=True)[:10]

print(f"[values] wrote {len(results)} rows to {OUTPUT_LOG.resolve()}")

fastest, slowest[:3]